<a href="https://colab.research.google.com/github/sanginnova8-hash/Ch-m-i-m-GVCN/blob/main/voice_clone_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Voice Clone cho project Fun Clone — F5-TTS trên Google Colab (Đã cấu hình sẵn)

Notebook này đã được cấu hình sẵn khớp với `voice_sample.mp3` và 157 đoạn thoại trong `02-VOICE-fun-clone-full.txt`.

**Cách chạy nhanh:**
1. Menu **Runtime → Change runtime type → T4 GPU** → **Save**.
2. Chạy lần lượt từ **Ô 1** đến **Ô 7**.
3. Ở **Ô 2**, tải lên file `voice_sample.mp3` (hoặc `ref.wav` đính kèm). Code sẽ tự động cắt bỏ âm thừa bị cụt ở đầu/cuối (`0.30s -> 4.80s`) và chuẩn hoá về `ref.wav` (24kHz Mono).
4. **Ô 3** và **Ô 4** đã điền sẵn `REF_TEXT` chuẩn và toàn bộ 157 đoạn thoại — bạn chỉ cần bấm chạy mà không cần tải lại file `.txt` hay gõ tay `REF_TEXT`.
5. **Ô 5** sinh 157 file `1.wav ... 157.wav` và **Ô 7** nén thành `voice-clone-fun-clone.zip` tải về máy.

In [1]:
# Ô 1 — Kiểm tra GPU + cài F5-TTS (lần đầu ~2-4 phút, tự tải model 1.35 GB ở ô 5)
!nvidia-smi -L
!pip install -q f5-tts
print('Đã cài xong F5-TTS.')

GPU 0: Tesla T4 (UUID: GPU-29a441df-9cbd-c2e4-0d82-b053eb86c9d9)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Đã cài xong F5-TTS.


In [2]:
# Ô 2 — Tải lên file voice_sample.mp3 (hoặc ref.wav)
from google.colab import files
import subprocess, os
from IPython.display import Audio, display

if os.path.exists('voice_sample.mp3'):
    src = 'voice_sample.mp3'
elif os.path.exists('ref.wav'):
    src = 'ref.wav'
else:
    up = files.upload()
    src = list(up.keys())[0]

# Nếu tải lên voice_sample.mp3 gốc (6.03s), tự cắt đoạn câu hoàn chỉnh (0.30s -> 4.80s) để tránh chữ bị cụt ở đầu/cuối
orig_dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries',
                 'format=duration', '-of', 'csv=p=0', src]))
if orig_dur > 5.5:
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-ss', '0.30', '-to', '4.80',
                    '-i', src, '-ar', '24000', '-ac', '1', 'ref.wav'], check=True)
elif src != 'ref.wav':
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', src, '-t', '30',
                    '-ar', '24000', '-ac', '1', 'ref.wav'], check=True)

dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries',
              'format=duration', '-of', 'csv=p=0', 'ref.wav']))
print(f'✅ File mẫu chuẩn: ref.wav — {dur:.2f} giây (24kHz Mono). Nghe thử:')
display(Audio('ref.wav'))

Saving ref.wav to ref.wav
✅ File mẫu chuẩn: ref.wav — 4.50 giây (24kHz Mono). Nghe thử:


In [3]:
# Ô 3 — Nội dung khớp từng chữ với đoạn chuẩn (0.30s -> 4.80s) của voice_sample.mp3
# Giúp F5-TTS không phải chạy Whisper ASR, tránh tràn GPU và không bị ngắt giữa chừng.
REF_TEXT = "Uh, let me explain. Turns out, the Great War really wasn't that great."
print('REF_TEXT =', repr(REF_TEXT))

REF_TEXT = "Uh, let me explain. Turns out, the Great War really wasn't that great."


In [4]:
# Ô 4 — Đọc 157 đoạn lời thoại (tự động nạp sẵn từ 02-VOICE-fun-clone-full.txt, không cần upload lại)
import os

DEFAULT_SEGMENTS = [
  "America is the richest and most powerful country in history.",
  "But America's rise to number one was actually quite unexpected.",
  "In fact, most of North America was ignored for more than a hundred years after it was discovered.",
  "Here's a map that was made right before Christopher Columbus discovered America.",
  "The Vikings said they had discovered an island somewhere over here, but no one realized that it was actually a continent.",
  "So, Columbus believed that he could find a trade route to Asia by sailing west. But actually, he found some islands in the Caribbean Sea.",
  "Columbus didn't find the spices he was expecting, but he still believed that he was close to India.",
  "So, he called the natives Indians, got some goodies, and took them back to the king and queen of Spain.",
  "Hey, your majesties, I just got back from Asia.",
  "Great. Where are the spices?",
  "Well, uh, we haven't found the spices yet, but they've got lots and lots of gold.",
  "Now you're talking, Columbus. Go back. Tell them all to be Christians and bring back that gold.",
  "A few years before Columbus claimed the West Indies for Spain, the Portuguese had discovered that they could reach Asia by sailing around Africa.",
  "So, Spain and Portugal drew a line right down the middle of the map.",
  "And they agreed that all the non-Christian lands on this side of the line would belong to Portugal and everything on the other side would belong to Spain.",
  "So the Portuguese sailed to Asia and built a huge trading empire and the Spanish got busy conquering their half of the world. All of the interesting parts of the Americas were down here where there was lots of gold, silver, and people.",
  "So that's where the Spanish went.",
  "Actually, the Spanish also sent an expedition into North America in 1539, but they didn't find gold and silver.",
  "So they mostly ignored all these useless lands up here in North America.",
  "The English actually sent an explorer to North America while the Spanish were still exploring the islands in the Caribbean.",
  "That was 16 years before the Spanish discovered Florida.",
  "But after that, the English waited almost 100 years before they tried to colonize America.",
  "England's first colony was established at Roanoke, North Carolina in 1585.",
  "But the same year, England got busy fighting a war with Spain, and that colony mysteriously disappeared.",
  "So they tried again at Jamestown, and this time it worked.",
  "Next, the pilgrims arrived in Plymouth.",
  "The pilgrims were strict Christians who outlawed bad stuff like gambling and celebrating Christmas.",
  "Soon, lots and lots of people were coming to America, and before long, the East Coast was covered with British colonies.",
  "Meanwhile, the French were busy claiming all the land in the middle of North America.",
  "Now, it's 1754, and it's time for a war to figure out who really owns what in America.",
  "New France has a population of about 60,000 people.",
  "So, they team up with some Indian tribes to help them fight. That's why it's called the French and Indian War.",
  "The British colonies have a population of more than a million people, and they also team up with some Indian tribes.",
  "Who do you think won?",
  "That's right, the British beat the French.",
  "At the same time, there was also a big war back in Europe.",
  "And when it was over, it was time to redraw the map.",
  "France gave half of Louisiana to Spain before the war was over to make sure Britain didn't get it.",
  "Then after the war, France had to give the other half of Louisiana to Britain.",
  "France also had to give Canada to Britain.",
  "Now Britain had lots of land in America, but Britain had also spent lots of money on the war. And now they needed money.",
  "Hey colonies, you know that expensive war we helped you win?",
  "Yeah, thanks for the help.",
  "Sure, no problem. By the way, we are going to start collecting lots and lots of taxes to pay for all the help we give you.",
  "What? You never help us. We're the ones who are always helping you. And how can you tax us if we don't get to vote?",
  "The colonists weren't happy with the new taxes.",
  "And the disagreement turned into a fight and from a fight into a revolution.",
  "On the 4th of July 1776, the Americans declared their independence from Great Britain.",
  "And that was the beginning of the United States of America, right? Not quite.",
  "It would be 5 more years before the Americans won their war of independence.",
  "And after they had won their freedom, they had a new problem. The government they had created was a complete mess.",
  "The 13 colonies had essentially become 13 independent countries that were only united in name.",
  "The new nation was failing, and it was failing fast.",
  "So the founding fathers wrote a new constitution and created an entirely new federal government that actually worked.",
  "The new constitution became official when nine states had ratified it.",
  "And in 1789, George Washington was elected the first president of the United States.",
  "But the USA was only this big and all the land over here belonged to Spain.",
  "So how did America get more land?",
  "Well, over in France, a guy named Napoleon seized power and started conquering Europe.",
  "Napoleon also wanted to build an empire in America.",
  "So he made a deal with Spain that would give Louisiana back to France.",
  "But the Americans needed to use the Mississippi River to send goods to and from other places.",
  "And they were afraid of what Napoleon might do if he controlled the river.",
  "So America asked France to sell New Orleans to them.",
  "You know my plans for a great empire in North America.",
  "But now I must invade Britain. We can build our empire in America after we defeat the British. But if the Americans discover our plans for America, they will support Britain, which could be fatal.",
  "But if we sell Louisiana to the Americans, they will be our friends forever, and Britain will be ours. It would be foolish to sell our largest territory, and the Americans only want New Orleans.",
  "Enough. I've made up my mind. We will sell it all. The money will fund our war with Britain.",
  "And so America bought Louisiana for $15 million.",
  "America had just doubled in size.",
  "And that was good because America's population was also almost doubling every 20 years.",
  "In those days, the average American family had seven children. And all those kids needed someplace to live.",
  "So lots of Americans moved west where there was lots and lots of unclaimed land.",
  "Does anyone own this land?",
  "Of course not.",
  "Great, because now it's mine.",
  "But these lands, my hunting grounds.",
  "That's not proper English. You're supposed to say these lands were my hunting grounds.",
  "In 1819, America bought Florida from Spain and agreed that this line right here would be the border between the United States and New Spain.",
  "Just 2 years later, Mexico won its independence from Spain, and America had a new neighbor.",
  "The northern territories of Mexico were sparsely populated and difficult to control.",
  "So the Mexican government invited Americans to settle in Texas.",
  "These settlers quickly outnumbered the Mexicans and before long the Texans broke free from Mexico and created their own independent republic.",
  "Yeehaw. We want to join the USA.",
  "Do you have slaves?",
  "Yes, we do. And we're proud of it.",
  "Well, sorry. We don't need any more slave states.",
  "Hey guys, I'm president now and this is ours.",
  "In 1845, the United States changed his mind and let the Republic of Texas join the Union.",
  "Disputes over the border of Texas led to a war with Mexico.",
  "At the end of the war, the United States paid Mexico $15 million and took the northern parts of Mexico that President Polk wanted.",
  "Just 5 years later, the United States also bought another piece of land from Mexico for $10 million.",
  "Meanwhile, the British agreed to let the Americans have all of the Oregon territory below this line, as long as the British could have everything above the line.",
  "And so in just 50 years of great luck and one ugly war with Mexico, America now stretched from sea to sea.",
  "And now there was nowhere left to expand. Unless...",
  "Hello, Uncle Sam. I found an island in the middle of the ocean. Can I claim it for the US?",
  "Hmm. We might need that for World War II. Does anyone live there?",
  "No, sir.",
  "Is it covered in bird droppings?",
  "Yes, sir.",
  "Congratulations, son. You're standing on US soil.",
  "Actually, sir, I wouldn't call this soil.",
  "By the way, the Guano Island Act is still a law.",
  "So, if you're tired of paying way too much for rent, go find yourself an island to occupy.",
  "Wow. Everything is great.",
  "Right after the Americans got California, word spread that gold had been discovered.",
  "And lots and lots of people rushed to California to help dig up the gold.",
  "Up in the Oregon territory, there was lots of free farmland.",
  "America was thriving and there was peace on all sides.",
  "But there was a big problem on the inside and that was the problem of slavery.",
  "In northern states, slavery was illegal and many considered it to be a great evil that must be stopped.",
  "In the south, slavery was an important part of the economy.",
  "The Americans had been arguing about slavery for decades, but everything came to a head when Abraham Lincoln was elected president in 1860.",
  "Many people in the south believed that Lincoln would abolish slavery.",
  "So most of the slave states separated from the United States and made their own country.",
  "That led to a big civil war. Lots of people died.",
  "The North won, the slaves were freed, and the United States was united again.",
  "I'm sorry to hear about the assassination of President Lincoln.",
  "Yeah, another assassin tried to kill me on the same night, but I survived the stabbing.",
  "That's terrible. I mean, it's terrible that you were stabbed, but I'm glad that you survived.",
  "By the way, the Tsar wants America to have Alaska. How much can you pay for it?",
  "I think we can do 5 million. Maybe 5.5 million, but not more.",
  "How about 7 million?",
  "Uh, sure.",
  "Great. I have approval for 7 million. We can make the treaty tomorrow.",
  "Can we do it tonight?",
  "I should have asked for more money.",
  "Great. Now America has another huge chunk of land.",
  "Now it just needs a few tropical islands. Hey, look at that.",
  "Some American businessmen have taken control of the Hawaiian Islands and they're asking the United States to annex them.",
  "But the president of the United States said no.",
  "He believed that America must be a just and law-abiding nation that never misuses its great power to take advantage of the weak.",
  "But others feared that if America didn't get involved in the affairs of Hawaii, another great power like Germany or Japan would come and take the islands and use them as a base to attack America in a future war.",
  "Should America mind its own business or should it mind its own interests?",
  "America's policy of staying out of other nations' conflicts was about to be tested because something was brewing in Cuba and it wasn't coffee.",
  "Cuba was in the middle of a war for independence.",
  "Almost all of Spain's other colonies had already won their independence from Spain after the American and French revolutions, but Cuba, Puerto Rico, and the Philippines were still colonies of Spain.",
  "The Cuban War for Independence was constantly in American newspapers, and most Americans sympathized with the revolutionaries.",
  "In 1898, after an American warship sank in a mysterious explosion, the US told Spain to leave Cuba or else.",
  "Spain responded by declaring war on the United States.",
  "Then the US quickly captured Cuba, Puerto Rico, and the Philippines, effectively bringing an end to what remained of the Spanish Empire that Christopher Columbus had begun.",
  "During the Spanish-American War, the US also annexed Hawaii, which was becoming an important naval base.",
  "After the war, the president was shot, and so the young and energetic Theodore Roosevelt unexpectedly became president.",
  "Hey, Colombia. Uh, we want to build a canal right here.",
  "Sure, just pay us $10 million.",
  "It's a deal.",
  "Actually, uh, we changed our mind. You can't build a canal.",
  "What? But we made a deal.",
  "Sorry.",
  "Hey, Panama. Welcome to the Club of Countries. We want to build a canal right here.",
  "Sure. Go ahead.",
  "While Roosevelt was president, he built up the second largest navy in the world and used this powerful navy to influence global politics.",
  "The Americans finished building the Panama Canal in 1914, allowing the powerful US Navy to quickly move between the Atlantic and Pacific oceans.",
  "America had become a world power, and World War I had just begun.",
  "If you want me to make more videos like this, please subscribe to let me know, and be sure to watch the next video right here.",
  "If you want to learn more about American history, I'll leave notes and clarifications in the description.",
  "See you next time."
]

if os.path.exists('02-VOICE-fun-clone-full.txt'):
    raw = open('02-VOICE-fun-clone-full.txt', encoding='utf-8').read()
    segments = [s.strip() for s in raw.replace('\n', ' ').split('|') if s.strip()]
    print(f'Đã đọc {len(segments)} đoạn từ file 02-VOICE-fun-clone-full.txt.')
else:
    segments = DEFAULT_SEGMENTS
    print(f'Đã nạp sẵn {len(segments)} đoạn từ bộ nhớ tích hợp trong notebook.')

print(f'Đoạn 1: {segments[0]}')
print(f'Đoạn 157: {segments[-1]}')

Đã nạp sẵn 157 đoạn từ bộ nhớ tích hợp trong notebook.
Đoạn 1: America is the richest and most powerful country in history.
Đoạn 157: See you next time.


In [ ]:
# Ô 5 — Sinh giọng clone cho toàn bộ 157 đoạn (đã sửa file_out -> file_wave theo API mới)
import os, time, subprocess
from f5_tts.api import F5TTS

if not os.path.exists('ref.wav') and os.path.exists('voice_sample.mp3'):
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', 'voice_sample.mp3',
                    '-ss', '0.30', '-to', '4.80', '-ar', '24000', '-ac', '1', 'ref.wav'], check=True)

os.makedirs('out', exist_ok=True)
if 'tts' not in globals():
    tts = F5TTS(model='F5TTS_v1_Base')

t0 = time.time()
for i, seg in enumerate(segments, 1):
    out = f'out/{i}.wav'
    if os.path.exists(out) and os.path.getsize(out) > 10_000:
        continue
    try:
        tts.infer(ref_file='ref.wav', ref_text=REF_TEXT, gen_text=seg,
                  file_wave=out, speed=1.0, remove_silence=False)
    except TypeError:
        tts.infer(ref_file='ref.wav', ref_text=REF_TEXT, gen_text=seg,
                  file_wave=out)
    if i % 10 == 0 or i == len(segments):
        print(f'{i}/{len(segments)} đoạn — {time.time()-t0:.0f}s')
print('🎉 XONG tất cả 157 đoạn!')

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 54.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

F5TTS_v1_Base/model_1250000.safetensors: reconstructing file:   0%|          |  0.00B / 1.35GB            

... (183 outputs left out)

Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Uh, let me explain. Turns out, the Great War really wasn't that great. 
gen_text 0 Just 5 years later, the United States also bought another piece of land from Mexico for $10 million.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:03<00:00,  3.24s/it]


Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Uh, let me explain. Turns out, the Great War really wasn't that great. 
gen_text 0 Meanwhile, the British agreed to let the Americans have all of the Oregon territory below this line, as long as the British could have everything above the line.


Generating audio in 1 batches...


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Ô 6 (tuỳ chọn) — Nghe thử vài đoạn trước khi tải về
from IPython.display import Audio, display
for i in (1, 9, 45):
    p = f'out/{i}.wav'
    if os.path.exists(p):
        print('Đoạn', i, ':', segments[i-1][:70])
        display(Audio(p))

In [ ]:
# Ô 7 — Nén toàn bộ và tải về máy. Gửi file zip này lại cho Muse để ghép vào video.
import shutil
from google.colab import files
shutil.make_archive('voice-clone-fun-clone', 'zip', 'out')
files.download('voice-clone-fun-clone.zip')
print('Đã tải voice-clone-fun-clone.zip')